In [19]:
# Imports
import json                                             # Parses and generates JSON data
import os                                               # Interfaces with the operating system
import pickle                                           # Serializes and deserializes Python object structures
import urllib.request                                   # Fetches data from URLs
import numpy as np                                      # Supports large, multi-dimensional arrays and matrices
import pandas as pd                                     # Data manipulation and analysis library
import geopandas as gpd                                 # Extends pandas for working with spatial data
import requests                                         # Makes HTTP requests
from shapely.geometry import Point, Polygon, LineString # Defines and manipulates geometric objects
import urllib.parse  
import matplotlib.colors as colors
# Global Variables
current_dir = os.getcwd()
data_cache_path = os.path.join(current_dir, 'data_cache')
arcgis_cache_path = os.path.join(data_cache_path, 'arcgis')

arcgis_sources = { 
    # Alphabetical by key
    'adalskipulag2040'      : 'https://luk.skipulag.is/server/rest/services/Stafraent_adalskipulag/MapServer/11',
    'adveitulagnir'         : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/lukor_overlay/MapServer/602',
    'dreifistodva_byggingar': 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/Einlinumynd/FeatureServer/73',
    'dreifistodvar'         : 'https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/317',
    'fasteignir'            : 'https://arcgis-s-9.or.is/server/rest/services/Anna%C3%B0/Fasteignir_hms/MapServer/0',
    'gotur'                 : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/grunnkortDynamic/MapServer/16',
    'haspennulagnir'        : 'https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/314',
    'ibuar'                 : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/LUKOR_FeatureLayers/FeatureServer/8',
    'lagspennulagnir'       : 'https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/315',
    'lagspennuskapar'       : 'https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/324',
    'landeignaskra'         : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/lukor_overlay/MapServer/228',
    'lodirRVK'              : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/lukor_overlay/MapServer/658',
    'maelar'                : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/LUKOR_FeatureLayers/FeatureServer/7',
    'rafbunadur'            : 'https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/328',
    'rafmagnsmaelar'        : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/lukor_overlay/MapServer/676',
    'spennar'               : 'https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/321',
    'strond'                : 'https://arcgis-s-5.or.is/arcgis/rest/services/lukor/grunnkortDynamic/MapServer/19',
    'tengiskapar'           : 'https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/329',
    
    # UtilityNetwork Rafmagn_cim (Feature Server) UPDATED Layers
    'UN_Taeki'              : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/100', # Elec. Devices
    'UN_Samstaeda'          : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/110', # Elec. Containers
    'UN_Tengi'              : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/120', # Elec. Ports
    'UN_Raflogn'            : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/130', # Elec. Lines
    'UN_Kerfisgrein'        : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/150', # Elec. Subnetwork
    'UN_Tengibunadur'       : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/900', # Elec. Misc. Devices
    'UN_Lagnaleidir'        : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/910', # Elec. Trenches
    'UN_Mannvirki'          : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/920', # Elec. Structures (Buildings)

    # UtilityNetwork Rafmagn_cim (Feature Server) UPDATED Tables
    'UN_Tengibunadar_itarupplysingar'   : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/10', # Elec. Misc. Dev. Info
    'UN_Mannvirki_itarupplysingar'      : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/11', # Elec. Struct. Info
    'UN_Tengi_itarupplysingar'          : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/12', # Elec. Port Info
    'UN_Hlutir'                         : 'https://arcgis-s-10.or.is/server/rest/services/UtilityNetwork/Rafmagn_cim/FeatureServer/13', # Unknown
}

# Load Cache Helper
def load_from_cache(filename, cache_path):
    """ Load the DataFrame from cache if available. """
    filepath = os.path.join(cache_path, f"{filename}.pickle")
    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    return None 

# Save Cache Helper
def save_to_cache(df, filename, cache_path):
    """ Save the DataFrame to cache. """
    if not os.path.exists(cache_path):
        os.makedirs(cache_path)
    filepath = os.path.join(cache_path, f"{filename}.pickle")
    with open(filepath, 'wb') as f:
        pickle.dump(df, f)

def filter_linedata(df):
    """
    1. Removes rows where HLUTVERK is 'Skinna' or 'Kvísl'.
    2. Drops unnecessary metadata columns.
    """
    if df is None or df.empty:
        print("⚠️ Input DataFrame is empty.")
        return df

    # --- STEP 1: ROW FILTERING (Remove Busbars/Links) ---
    # We prefer the DECODED column for readable filtering, but fall back to raw
    filter_col = 'HLUTVERK_DECODED' if 'HLUTVERK_DECODED' in df.columns else 'HLUTVERK'
    
    garbage_types = ['Skinna', 'Kvísl']
    
    # Keep only rows that are NOT in the garbage list
    clean_df = df[~df[filter_col].isin(garbage_types)].copy()
    
    print(f"🧹 Removed {len(df) - len(clean_df)} rows (Skinna/Kvísl).")

    # --- STEP 2: COLUMN DROPPING (Remove Metadata) ---
    cols_to_remove = [
        "SKRASETJARI", 
        "MUFFUKERFI", 
        "MUFFUKERFI_DECODED", 
        "LAST_EDITED_USER", 
        "HLUTVERK_DECODED", # We drop this AFTER using it for the row filter
        "GAGNAEIGANDI",
        "FSVEFUR",
        "FM_INN_DECODED",
        "FM_INN",
        "FM_BREYTT_DECODED",
        "FM_BREYTT", 
        "EIGANDI_DECODED", 
        "DMM_HLEKKUR",
        "DMM_LYKILL",
        "CREATED_USER",
        "VINNSLUFERLIFITJU",
        "VIDMIDUNPLANUPPR",
        "VERKNR",
        "TEGUND",
        "SVF",
        "STADA",
        "SPENNA_DECODED",
        "VIDMIDUNPLANUPPR_DECODED",
        "HEIMILD",
        "MALTILV",
        "FLOKKUR_IST120"
        "GERD"

    ]
    
    # errors='ignore' prevents crashing if a column is already missing
    clean_df = clean_df.drop(columns=cols_to_remove, errors='ignore')
    
    print(f"🗑️ Dropped {len(cols_to_remove)} metadata columns.")
    return clean_df
# ArcGIS Data Fetcher


def get_arcgis_data(source, where_clause="1=1", cache_data=False, sort_columns=False, verbose=False):
    """
    MODIFIED: Now accepts 'where_clause' to filter data (e.g., "DNR = 659").
    Default is "1=1" (fetch everything) to maintain backward compatibility.
    """

    # Helper function for checking if the manual url is valid
    def is_valid_arcgis_url(url):
        try:
            response = requests.get(f"{url}?f=pjson")
            response.raise_for_status()
            data = response.json()
            return 'fields' in data and 'type' in data
        except requests.RequestException:
            return False

    # Helper function for decoding types
    def decode_by_types(queried_df, metadata, type_field):
        type_field = type_field.upper()
        types_name_mappings = {}
        types_domains_mappings = {}
        for type_dict in metadata['types']:
            types_name_mappings[type_dict['id']] = type_dict['name']
            types_domains_mappings[type_dict['id']] = {}
            if 'domains' in type_dict and type_dict['domains'] is not None:
                for attribute in type_dict['domains'].keys():
                    if 'codedValues' in type_dict['domains'][attribute].keys():
                        types_domains_mappings[type_dict['id']][attribute] = {
                            entry['code']: entry['name'] for entry in type_dict['domains'][attribute]['codedValues']
                        }
        
        if type_field in queried_df.columns:
            queried_df[f'{type_field}_DECODED'] = queried_df[type_field].map(lambda x: types_name_mappings.get(x, x))

        df_slices = []
        for type_value, group_df in queried_df.groupby(type_field):
            type_domain_mappings = types_domains_mappings.get(type_value, {})
            for attribute, mapping in type_domain_mappings.items():
                if attribute in group_df.columns:
                    decoded_col_name = f'{attribute}_DECODED'
                    group_df[decoded_col_name] = group_df[attribute].map(lambda x: mapping.get(x, x))
            df_slices.append(group_df)

        if df_slices: 
            queried_df = pd.concat(df_slices)
        return queried_df

    # --- MAIN LOGIC STARTS HERE ---
    
    # 1. Skip Cache if we are using a specific filter
    # (We usually don't want to cache partial datasets under the generic name)
    need_query = True
    if where_clause == "1=1" and cache_data:
        cached_data = load_from_cache(source, cache_path=arcgis_cache_path)
        if cached_data is not None:
            if verbose: print(f"Loaded data from cache for: {source}")
            return cached_data
        else:
            if verbose: print(f"No data loaded from cache for: {source}")
    
    # 2. Setup URL and Filter
    base_url = arcgis_sources.get(source, source)
    if base_url not in arcgis_sources.values() and not is_valid_arcgis_url(base_url):
        raise ValueError("Provided source key is not a valid predefined key or a valid ArcGIS layer URL.")
    
    # Safe Encode the filter (e.g., "DNR = 659" -> "DNR%20%3D%20659")
    safe_where = urllib.parse.quote(where_clause)

    # 3. Fetch Metadata
    json_url = f'{base_url}?f=pjson'
    response = urllib.request.urlopen(urllib.request.Request(json_url)).read()
    metadata = json.loads(response.decode('utf-8'))

    if 'fields' not in metadata or metadata['fields'] is None:
        return None

    # 4. Get Record Count (APPLYING FILTER)
    count_url = (
        f"{base_url}/query?"
        f"where={safe_where}&"  # <--- CHANGED FROM 1=1
        "returnCountOnly=true&"
        "f=pjson"
    )
    
    if verbose: print(f"Fetching record count for query: {where_clause}")
    response = urllib.request.urlopen(urllib.request.Request(count_url)).read()
    total_records = json.loads(response.decode('utf-8')).get('count', 0)
    
    if verbose: print(f"Total records found: {total_records}")

    if total_records > 0:
        batch_size = int(0.8 * metadata.get('maxRecordCount', 1000))
        offset = 0
        features = []

        # 5. Fetch Data Batches (APPLYING FILTER)
        while offset < total_records:
            batch_url = (
                f"{base_url}/query?"
                f"where={safe_where}&" # <--- CHANGED FROM 1=1
                "geometryType=esriGeometryEnvelope&"
                "spatialRel=esriSpatialRelIntersects&"
                "returnGeometry=true&"
                "returnTrueCurves=false&"
                "returnIdsOnly=false&"
                "returnCountOnly=false&"
                "returnZ=false&"
                "returnM=false&"
                "returnDistinctValues=false&"
                f"resultOffset={offset}&"
                f"resultRecordCount={batch_size}&"
                "returnExtentsOnly=false&"
                "f=pjson&"
                "outFields=*&"
                "orderByFields=OBJECTID"
            )
            
            if verbose: print(f"Fetching batch offset={offset}")
            response = urllib.request.urlopen(urllib.request.Request(batch_url)).read()
            batch_data = json.loads(response.decode('utf-8'), strict=False)
            
            if 'features' in batch_data:
                features += batch_data['features']
                offset += len(batch_data['features'])
            else:
                break # Stop if no features returned

        # 6. Create DataFrame & Convert Types (Original Logic)
        attributes = [feature['attributes'] for feature in features]
        queried_df = pd.DataFrame(attributes)

        for field in metadata['fields']:
            if field['type'] in ('esriFieldTypeInteger', 'esriFieldTypeSmallInteger'):
                if field['name'] in queried_df.columns:
                    queried_df[field['name']] = pd.to_numeric(queried_df[field['name']]).astype('Int64')

        # 7. Construct Geometry (Original Logic)
        if 'geometryType' in metadata:
            if metadata['geometryType'] == 'esriGeometryPoint':
                geometry = [(Point(x['geometry']['x'], x['geometry']['y']) if ('geometry' in x and 'x' in x['geometry']) else None) for x in features]
            elif metadata['geometryType'] == 'esriGeometryPolygon':
                geometry = [(Polygon(np.concatenate(x['geometry']['rings'])) if ('geometry' in x and 'rings' in x['geometry']) else None) for x in features]
            else: # Polyline
                geometry = [(LineString(np.concatenate(x['geometry']['paths'])) if ('geometry' in x and 'paths' in x['geometry']) else None) for x in features]

            try:
                queried_df = gpd.GeoDataFrame(queried_df, geometry=geometry, crs=f'EPSG:{metadata["sourceSpatialReference"]["wkid"]}')
            except Exception:
                # Fallback to extent CRS if source fails
                 if "extent" in metadata and "spatialReference" in metadata["extent"]:
                    queried_df = gpd.GeoDataFrame(queried_df, geometry=geometry, crs=f'EPSG:{metadata["extent"]["spatialReference"]["wkid"]}')

        # 8. Decode Fields (Original Logic)
        field_domain_mappings = {}
        for field in metadata['fields']:
            if 'domain' in field and field['domain']:
                if 'codedValues' in field['domain']:
                    field_domain_mappings[field['name']] = {
                        entry['code']: entry['name'] for entry in field['domain']['codedValues']
                    }
        
        for col, map_dict in field_domain_mappings.items():
            if col in queried_df.columns:
                queried_df[f'{col}_DECODED'] = queried_df[col].map(lambda x: map_dict.get(x, x))

        # 9. Decode Types (Original Logic)
        if 'types' in metadata:
            type_field = None
            for field in metadata['fields']:
                if field['name'] == 'ASSETGROUP':
                    type_field = 'ASSETGROUP'
                    break
                elif field['name'] == 'TIERNAME':
                    type_field = 'TIERNAME'
                    break
            
            if type_field:
                queried_df = decode_by_types(queried_df, metadata, type_field)

        # 10. Specific Logic for Lágspennulagnir (Original Logic)
        if source == 'lagspennulagnir' and 'STRBUTUR' in queried_df.columns:
            # Wrap in try/except in case STRBUTUR is empty or format varies
            try:
                extracted = queried_df['STRBUTUR'].str.extract(r"(?P<FROM>\w*\d+)-(?P<TO>\w*\d+)(?P<SEP>\w)(?P<SECTION>\d+)")
                queried_df = pd.concat([queried_df, extracted], axis=1)
                queried_df['SECTION'] = pd.to_numeric(queried_df['SECTION']).astype('Int32')
            except Exception as e:
                if verbose: print(f"Note: Could not parse STRBUTUR: {e}")

        # Sort columns
        if sort_columns:
            queried_df = queried_df[sorted(queried_df.columns)]

        # Cache only if it was a full download (1=1) and using a known source key
        if cache_data and where_clause == "1=1":
            if source in arcgis_sources:
                save_to_cache(queried_df, source, cache_path=arcgis_cache_path)

        return queried_df

    else:
        if verbose: print(f'No records found for query: {where_clause}')
        field_names_dict = {field['name']: None for field in metadata['fields']}
        return pd.DataFrame([field_names_dict])

In [3]:
import requests
import pandas as pd
import json

def fetch_data_by_dnr(substation_number, layer_id):
    """
    Fetches data from the Rafmagn MapServer for a specific substation (DNR)
    and a specific layer.
    """
    base_url = "https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer"
    query_url = f"{base_url}/{layer_id}/query"
    
    # The 'where' parameter is the SQL filter for the server
    params = {
        "where": f"DNR = {substation_number}",
        "outFields": "*",          # Fetch all fields (attributes)
        "returnGeometry": "true",  # Set to true if you need the coordinates/shapes
        "f": "json"                # Format response as JSON
    }
    
    print(f"Querying Layer {layer_id} for DNR: {substation_number}...")
    
    try:
        response = requests.get(query_url, params=params)
        response.raise_for_status() # Check for HTTP errors
        data = response.json()
        
        # Check if any features were returned
        if 'features' in data and len(data['features']) > 0:
            count = len(data['features'])
            print(f"✅ Found {count} items in Layer {layer_id}")
            
            # OPTIONAL: Convert attributes to a clean DataFrame for viewing
            # This flattens the 'attributes' dictionary from the JSON
            attributes_list = [f['attributes'] for f in data['features']]
            df = pd.DataFrame(attributes_list)
            return df
        else:
            print(f"⚠️ No data found in Layer {layer_id} for DNR {substation_number}")
            return None
            
    except Exception as e:
        print(f"❌ Error querying layer {layer_id}: {e}")
        return None

# --- MAIN EXECUTION ---

# 1. Define the Substation ID you want (e.g., 659 or 411)
target_dnr = 659

# 2. Define the layers you want to check. 
# You identified 324 (Lágspennulagnir). 
# You will likely need to find the IDs for Spennar (Transformers) and Skerar (Cabinets).
layer_ids_to_check = [315] # 315 for lágspennulagnir, add more layer IDs as needed

# 3. Loop through layers and fetch data
results = {}
for layer in layer_ids_to_check:
    df = fetch_data_by_dnr(target_dnr, layer)
    if df is not None:
        results[layer] = df

# Example: Display the first few rows of the data from layer 324
if 324 in results:
    print("\nData for Layer 324 (Lágspennulagnir):")
    print(results[324].head())

Querying Layer 315 for DNR: 659...
✅ Found 775 items in Layer 315


In [4]:
target_dnr = 659
filter_query = f"DNR = {target_dnr}"

sample_df = get_arcgis_data(source='https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer/315',where_clause=filter_query,sort_columns=True, verbose=True)
sample_df = filter_linedata(sample_df)
sample_df.head()

Fetching record count for query: DNR = 659
Total records found: 775
Fetching batch offset=0
🧹 Removed 318 rows (Skinna/Kvísl).
🗑️ Dropped 26 metadata columns.


,AFANGASTADUR,ATHUGASEMDIR,BUN_FRA,BUN_TIL,CREATED_DATE,DAGSHEIMILDAR,DAGSINNSETNINGAR,DAGSLEIDRETTINGAR,DNR,DROFI,...,STRIKAMERKI,SVF_DECODED,SYMBOL,TEGUND_DECODED,TENGD,TENGD_DECODED,TO_DEVICE_TERMINAL,VBL,VINNSLUFERLIFITJU_DECODED,geometry
0,None,None,<NA>,<NA>,1.641993e+12,None,1630585250000,1664203855000,659,0,...,None,Reykjavík,98,Lágspennustrengur,1,Já,16,<NA>,NaN,"LINESTRING (362889.623 407142.692, 362890.314 ..."
1,None,None,0,12539,NaN,None,662515200000,1589376694000,659,1,...,None,Reykjavík,4,Lágspennustrengur,1,Já,0,1550614,Skjáhnitun/Borðhnitun,"LINESTRING (362896.99 406780.056, 362896.495 4..."
2,None,None,0,12521,NaN,None,662515200000,1589378183000,659,6,...,None,Reykjavík,4,Lágspennustrengur,1,Já,0,1550665,Skjáhnitun/Borðhnitun,"LINESTRING (362746.91 406985.512, 362780.528 4..."
3,None,None,<NA>,<NA>,1.320060e+12,None,1320059596000,1320061861000,659,-,...,None,Reykjavík,4,Lágspennustrengur,1,Já,0,1065444,Skjáhnitun/Borðhnitun,"LINESTRING (363339.324 407319.333, 363346.13 4..."
4,None,None,<NA>,<NA>,1.320060e+12,None,1320059884000,1320061861000,659,-,...,None,Reykjavík,4,Lágspennustrengur,1,Já,0,1065481,Skjáhnitun/Borðhnitun,"LINESTRING (363358.988 407396.053, 363365.333 ..."


In [22]:

# --- 1. CONFIGURATION ---
target_dnr = 1354
filter_query = f"DNR = {target_dnr}"

# --- 2. FETCH DATA ---
print(f"📡 Fetching data for Substation {target_dnr}...")

# Fetch raw data (Robust Fetcher)
lines_raw = get_arcgis_data('lagspennulagnir', where_clause=filter_query)
transformers = get_arcgis_data('spennar', where_clause=filter_query)
cabinets = get_arcgis_data('tengiskapar', where_clause=filter_query)

# --- 3. SEPARATE "VALID" VS "JUNK" ---
# Apply your filter to get the clean lines
lines_clean = filter_linedata(lines_raw)

# Find the "Junk" lines by looking at what was removed
# (Indices in Raw that are NOT in Clean)
junk_lines = lines_raw[~lines_raw.index.isin(lines_clean.index)]

print(f"📊 Statistics:")
print(f"   - Valid Cables: {len(lines_clean)}")
print(f"   - Excluded Junk (Skinna/Kvísl): {len(junk_lines)}")
print(f"   - Cabinets: {len(cabinets)}")
print(f"   - Transformers: {len(transformers)}")



# 1. Define your exact color mapping
# Keys must match the values in the 'HLUTVERK' column exactly
my_color_map = {
    "Lágspennudreifilögn": "#32CD32",  # Bright Green (Lime)
    "Götuljósalögn": "#FF8C00",        # Dark Orange
    "Heimtaug": "#0000FF",             # Blue
}

# 2. Create a dynamic color list based on the data present
# Geopandas sorts the unique values alphabetically when assigning colors.
# We must build our color list in that SAME order.
unique_values = sorted(lines_clean['HLUTVERK'].unique())

# Get the color for each value (use 'gray' for anything not in your map)
cmap_colors = [my_color_map.get(val, 'gray') for val in unique_values]

# 3. Create the Matplotlib Colormap
custom_cmap = colors.ListedColormap(cmap_colors)

# 4. Plot using the custom cmap
m = lines_clean.explore(
    column='HLUTVERK',          # The column to color by
    cmap=custom_cmap,           # Your custom color list
    tooltip=['GERD_DECODED', 'SHAPE.LEN', 'HLUTVERK'],
    name="Valid Cables",
    style_kwds={'weight': 4}
)

# 2. Add Transformers (Red)
# We pass 'm=m' to add it to the EXISTING map
if transformers is not None:
    transformers.explore(
        m=m, 
        color='red',
        marker_kwds={'radius': 8},
        tooltip=['NUMER', 'MALRAUN', 'FRAMLEIDANDI'], 
        name="Transformers"
    )

# 3. Add Cabinets (Black)
if cabinets is not None:
    cabinets.explore(
        m=m, 
        color='black',
        marker_kwds={'radius': 4},
        tooltip=['TENGINR', 'GERD', 'STADA'], 
        name="Cabinets"
    )

# 4. Show it
m

📡 Fetching data for Substation 1354...
🧹 Removed 64 rows (Skinna/Kvísl).
🗑️ Dropped 26 metadata columns.
📊 Statistics:
   - Valid Cables: 96
   - Excluded Junk (Skinna/Kvísl): 64
   - Cabinets: 6
   - Transformers: 2


In [7]:
print(df['HLUTVERK'].unique())

['Lágspennudreifilögn' 'Götuljósalögn' 'Heimtaug' 'Skinna' 'Kvísl']


In [8]:
# Print the count of each unique entry in 'HLUTVERK'
print(df['HLUTVERK'].value_counts())

# Example: Print all rows where HLUTVERK equals a specific value (replace 'x' with the desired value)
x = 'Straumspennir'  # Change this to any value from df['HLUTVERK'].unique()
df[df['HLUTVERK'] == x].head()

HLUTVERK
Skinna                 315
Heimtaug               235
Götuljósalögn          173
Lágspennudreifilögn     49
Kvísl                    3
Name: count, dtype: int64


,OBJECTID,HLUTVERK,TEGUND,GERD,DNR,STRBUTUR,NR,SPENNA,TENGD,MALTILV,...,SKRASETJARI,SKRAD_DAGS,MUFFUKERFI,DMM_HLEKKUR,TO_DEVICE_TERMINAL,FROM_DEVICE_TERMINAL,STRIKAMERKI,ATHUGASEMDIR,FASI,SHAPE.LEN


In [9]:
import requests
import pandas as pd

# Set pandas to show all columns (vital for inspecting raw data)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

def fetch_raw_layer(layer_id, layer_name, dnr):
    base_url = "https://arcgis-s-5.or.is/arcgis/rest/services/Rafmagn/Rafmagn/MapServer"
    query_url = f"{base_url}/{layer_id}/query"
    
    params = {
        "where": f"DNR = {dnr}",
        "outFields": "*",          # Grab EVERYTHING
        "returnGeometry": "true",  # We need coordinates later
        "f": "json"
    }
    
    print(f"--- Fetching {layer_name} (ID: {layer_id}) for DNR: {dnr} ---")
    
    try:
        response = requests.get(query_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'features' in data and len(data['features']) > 0:
            # Flatten the attributes and geometry into a single dataframe
            rows = []
            for feature in data['features']:
                row = feature['attributes']
                # Add geometry explicitly so we don't lose it
                if 'geometry' in feature:
                    row.update(feature['geometry']) 
                rows.append(row)
            
            df = pd.DataFrame(rows)
            print(f"✅ Found {len(df)} records.\n")
            return df
        else:
            print(f"⚠️ No data found.\n")
            return None
            
    except Exception as e:
        print(f"❌ Error: {e}\n")
        return None

# ==========================================
# MAIN EXECUTION
# ==========================================

# 1. Set your Substation ID
TARGET_DNR = 659  

# 2. Define the layers based on your text
layers = [
    (315, "Lágspennulagnir (low voltage Lines)"),
    (321, "Spennir (Transformers)"),
    (329, "Tengiskápar (Cabinets)")
]

raw_data = {}

# 3. Fetch and Display
for layer_id, name in layers:
    df = fetch_raw_layer(layer_id, name, TARGET_DNR)
    if df is not None:
        raw_data[layer_id] = df
        # Print the first 2 rows to see the column names and sample data
        print(f"SAMPLE DATA FOR {name}:")
        print(df.head(2))
        print("-" * 80 + "\n")

--- Fetching Lágspennulagnir (low voltage Lines) (ID: 315) for DNR: 659 ---
✅ Found 775 records.

SAMPLE DATA FOR Lágspennulagnir (low voltage Lines):
    OBJECTID             HLUTVERK             TEGUND  GERD  DNR        STRBUTUR   NR  SPENNA  TENGD MALTILV  LAGNINGARAR  HNUTUR  DSPENNIR  DSKAPUR DROFI  HLENGD EIGANDI  LEGA   SVF    VERKNR        VBL  SYMBOL HEIMILD DAGSHEIMILDAR FM_INN FM_BREYTT  DAGSINNSETNINGAR  DAGSLEIDRETTINGAR           GAGNAEIGANDI  NAKVAEMNIXY  VINNSLUFERLIFITJU  VIDMIDUNPLANUPPR  FLOKKUR_IST120 KOST_SV LENGD  SHAPE_LENGD                                            FSVEFUR  OBJECT_ID  BUN_TIL  BUN_FRA                                GLOBALID    STADA              CREATED_USER  CREATED_DATE LAST_EDITED_USER  LAST_EDITED_DATE  DMM_LYKILL  STAERD AFANGASTADUR GOTULJOSAKERFI  SPENNUSETT_DAGS SKRASETJARI  SKRAD_DAGS MUFFUKERFI                                        DMM_HLEKKUR  TO_DEVICE_TERMINAL  FROM_DEVICE_TERMINAL STRIKAMERKI ATHUGASEMDIR FASI  SHAPE.LEN         

In [10]:
raw_data[321].head() # Transformers

,OBJECTID,DNR,NUMER,FRAMLEIDANDI,FRAMLEIDSLUNR,FRAMLEIDSLUAR,GERD,OLIAUNDIRGERD,NETTOTHYNGD,MAGN_OLIU,TEG_OLIU,MALRAUN,TENGIFLOKKUR,FJOLDI_FASA,FORSPENNA,EFTIRSPENNA,SPENNUSTILLING,SVID_SPENNUSTILLINGAR,ONNUR_EFTIRSPENNA,SKAMMHLAUPSPENNA,FORSTRAUMUR,EFTIRSTRAUMUR,HITAMAELIR,SKAPURHITAMAELA,LOKADAR_TENGINGAR,KENNI,SNUNINGUR,EIGANDI,SVF,VERKNR,VBL,FSVEFUR,HEIMILD,DAGSHEIMILDAR,STADA,FM_INN,FM_BREYTT,DAGSINNSETNINGAR,DAGSLEIDRETTINGAR,GAGNAEIGANDI,NAKVAEMNIXY,NAKVAEMNIZ,VINNSLUFERLIFITJU,VIDMIDUNPLANUPPR,FLOKKUR_IST120,GLOBALID,SKRASETJARI,DMM_LYKILL,X,Y,ID,TYPE,SYMBOL,ARTICLE,DMM_HLEKKUR,x,y
0,4161297,659,1,Rafha,TB2160,1998,Olíuspennir með aukageymi,None,1500.0,450.0,Jarðefnaolía,500,Dyn5,3,11000,400,Utaná,"±2x2,5%",None,5.1,26.2,722.0,Með mestvísun,Nei,Já,D0659_S-11-01,164.887,Veitur,None,None,None,None,FieldMaps,1694172171000,Innfært,AB,MT,1630585250000,1664122593000,Orkuveita Reykjavíkur,None,None,4,1,None,{07A45E3C-BAEF-4B2C-9DD8-F8971F181BAC},rakelao@OR,793,362891.2,407142.2,659-1,SP500E. T,416,None,https://dmm.veitur.is/handlers/legacy/eign_opi...,362891.2448,407142.2304
1,4161270,659,2,MÖRE TRAFO,0700447,2007,Olíuspennir án aukageymis,None,1765.0,458.0,Jarðefnaolía,800,Dyn5,3,11000,400,Utaná,"±2x2,5%",None,5.7,42.0,1154.7,Með mestvísun,Nei,Já,D0659_S-11-02,164.887,Veitur,None,None,None,None,FieldMaps,1694172071000,Innfært,AB,MT,1630585250000,1664122546000,Orkuveita Reykjavíkur,None,None,4,1,None,{45457633-2D03-47DB-A2E4-0A6370B2042A},rakelao@OR,794,362891.5,407143.1,659-2,0TW6960,416,None,https://dmm.veitur.is/handlers/legacy/eign_opi...,362891.4795,407143.0993


In [11]:
raw_data[315].head() # line data

,OBJECTID,HLUTVERK,TEGUND,GERD,DNR,STRBUTUR,NR,SPENNA,TENGD,MALTILV,LAGNINGARAR,HNUTUR,DSPENNIR,DSKAPUR,DROFI,HLENGD,EIGANDI,LEGA,SVF,VERKNR,VBL,SYMBOL,HEIMILD,DAGSHEIMILDAR,FM_INN,FM_BREYTT,DAGSINNSETNINGAR,DAGSLEIDRETTINGAR,GAGNAEIGANDI,NAKVAEMNIXY,VINNSLUFERLIFITJU,VIDMIDUNPLANUPPR,FLOKKUR_IST120,KOST_SV,LENGD,SHAPE_LENGD,FSVEFUR,OBJECT_ID,BUN_TIL,BUN_FRA,GLOBALID,STADA,CREATED_USER,CREATED_DATE,LAST_EDITED_USER,LAST_EDITED_DATE,DMM_LYKILL,STAERD,AFANGASTADUR,GOTULJOSAKERFI,SPENNUSETT_DAGS,SKRASETJARI,SKRAD_DAGS,MUFFUKERFI,DMM_HLEKKUR,TO_DEVICE_TERMINAL,FROM_DEVICE_TERMINAL,STRIKAMERKI,ATHUGASEMDIR,FASI,SHAPE.LEN,paths
0,247004217,Lágspennudreifilögn,Lágspennustrengur,42.0,659,12526-12536A01,0.0,400.0,1,,1986,1.0,2.0,1.0,1,74.7,Veitur,6,0000,70002525,1550631.0,2,,None,BRS,ÁJG,662515200000,1665567581000,Orkuveita Reykjavíkur,10.0,4.0,NaN,403.0,1,None,74.569616,https://orkuveitareykjavikur.sharepoint.com/si...,33813,12536.0,12526.0,{39533D64-3BCA-467E-8C6B-FE89C05AF021},Innfært,Berglind R. Sveinsdóttir,NaN,ALFHEIDUR1,1.665568e+12,33813.0,NaN,None,None,NaN,ANNA,NaN,None,https://dmm.veitur.is/handlers/legacy/eign_opi...,4.0,4.0,None,None,RST,74.569616,"[[[362882.5204375014, 406962.14493750036], [36..."
1,247004218,Lágspennudreifilögn,Lágspennustrengur,42.0,659,12536-12537A01,0.0,400.0,1,,1986,1.0,2.0,1.0,1,92.2,Veitur,6,0000,70002525,1550626.0,2,,None,BRS,ÁJG,662515200000,1665567580000,Orkuveita Reykjavíkur,10.0,4.0,NaN,403.0,1,None,92.064743,https://orkuveitareykjavikur.sharepoint.com/si...,845,12537.0,12536.0,{8925EFC2-40F0-45B3-982E-E97173B33FA9},Innfært,Berglind R. Sveinsdóttir,NaN,ALFHEIDUR1,1.665568e+12,845.0,NaN,None,None,NaN,ANNA,NaN,None,https://dmm.veitur.is/handlers/legacy/eign_opi...,4.0,4.0,None,None,RST,92.064743,"[[[362864.7341874987, 406895.5304374993], [362..."
2,246811675,Lágspennudreifilögn,Lágspennustrengur,371.0,659,D0659_BB-STR-1,NaN,NaN,1,None,2021,NaN,1.0,1.0,0,NaN,Veitur,20,0000,None,NaN,98,None,None,AB,MT,1630585250000,1664203855000,Orkuveita Reykjavíkur,NaN,NaN,NaN,NaN,None,None,3.464201,None,567825,NaN,NaN,{849DBDB3-219C-4508-B1C0-54F7C6F67CA7},Innfært,JAKOB,1.641993e+12,MARIA,1.664204e+12,NaN,NaN,None,None,NaN,None,NaN,None,None,16.0,1.0,None,None,None,3.464201,"[[[362889.6228125021, 407142.69249999896], [36..."
3,246816855,Götuljósalögn,Lágspennustrengur,31.0,659,0-12539A01,0.0,400.0,1,' ',1992,1.0,2.0,2.0,1,115.6,Reykjavíkurborg,6,0000,70002525,1550614.0,4,' ',None,BRS,ABA,662515200000,1589376694000,Orkuveita Reykjavíkur,10.0,4.0,NaN,403.0,1,None,39.273139,https://orkuveitareykjavikur.sharepoint.com/si...,46219,12539.0,0.0,{555F1BEB-3959-4573-AB67-BB3EE746719C},Innfært,Berglind R. Sveinsdóttir,NaN,ANNA,1.586851e+12,46219.0,NaN,None,None,NaN,ANNA,NaN,None,None,0.0,0.0,None,None,None,39.273139,"[[[362896.9900000021, 406780.0560000017], [362..."
4,246817395,Götuljósalögn,Lágspennustrengur,31.0,659,0-12521A01,0.0,400.0,1,,1991,1.0,1.0,1.0,6,150.2,Reykjavíkurborg,6,0000,70002525,1550665.0,4,,None,BRS,ABA,662515200000,1589378183000,Orkuveita Reykjavíkur,10.0,4.0,NaN,403.0,1,None,38.736804,https://orkuveitareykjavikur.sharepoint.com/si...,92577,12521.0,0.0,{8FAB9C4D-5EAB-4A2E-87B4-468F5D402A97},Innfært,Berglind R. Sveinsdóttir,NaN,None,NaN,92577.0,NaN,None,None,NaN,None,NaN,None,None,0.0,0.0,None,None,None,38.736804,"[[[362746.9099999964, 406985.51199999824], [36..."


In [12]:
raw_data[329].head() # cabinet data

,OBJECTID,HLUTVERK,GERD,DNR,GLDNR,TENGINR,KERFI,MALTILV,EIGANDI,LAGNINGARAR,SVF,VERKNR,VBL,SYMBOL,SNUNINGUR,HEIMILD,DAGSHEIMILDAR,FM_INN,FM_BREYTT,DAGSINNSETNINGAR,DAGSLEIDRETTINGAR,GAGNAEIGANDI,NAKVAEMNIXY,NAKVAEMNIZ,VINNSLUFERLIFITJU,VIDMIDUNPLANUPPR,FLOKKUR_IST120,KOST_SV,KENNI,STAERD,FSVEFUR,DSPENNIR,DSKAPUR,DROFI,STADUR,PNR,SK_FJARLAEGD_STAD,SK_FJOLDI_MODULA,SK_SKINNUSTAERD,SK_HAED,SK_BREIDD,SK_DYPT,SK_THYNGD,SK_LAUS_PLASS,SK_STADA,SK_UPPSETTUR_AF,SK_UPPSETTUR_DAGS,SK_MYND_AFSTODU_DAGS,SK_MYND_DAGS,SK_UPPF_UR_SKAPASKRA,X,Y,GLOBALID,STADA,TEGUND,GOTULJOSASKAPUR_STYRING,ALAGSSTYRING_RAS,MAELIR,SKAMMHLAUPSV_RAFHLADA,DMM_LYKILL,SKREFSPENNUSKAUT,SOKKULSKAUT,SERSKAUT,JARDVIDNAM,JARDVIDNAM_MAELT,CREATED_USER,CREATED_DATE,LAST_EDITED_USER,LAST_EDITED_DATE,ENT_STODVARNOTKUN,ENT_LEKALIDI_STAERD,ENT_LEKALIDI_GERD,ENT_SJALFVOR_STAERD,ENT_SJALFVOR_GERD,FJSK_RTU,FJSK_UPS,FJSK_SWITCH,FJSK_LJOSBREYTA,FJSK_FJARSKIPTI,FJSK_STADA,DMM_GATLISTI,DMM_HLEKKUR,STRIKAMERKI,x,y
0,324289766,Tengiskápur,KSIP 443,659,659.0,12526,Notendakort,None,Veitur,1987,0000,70002525,1550643.0,91,167.0,None,None,BRS,ABA,762998400000,1589377965000,Orkuveita Reykjavíkur,10.0,None,4,NaN,403.0,1,None,None,https://orkuveitareykjavikur.sharepoint.com/si...,2,1,1,Gerðhamrar 1,112,19.477754,43.0,None,1200.0,610.0,210.0,45.0,11,Skráður,EV,-2208988800000,1588710099000,1588710099000,"Nei, LUKOR",362882.557,406962.238,{0459C322-15DC-4A80-963E-783D6C83F949},Innfært,None,None,None,None,None,4467,None,None,None,None,None,None,NaN,ANNA,1.589378e+12,None,None,None,None,None,None,None,None,None,None,None,None,http://dmm.veitur.is/handlers/legacy/eign_opin...,E.25-12342,362882.557,406962.238
1,324290597,Tengiskápur,KSIP 433,659,659.0,12509,Notendakort,None,Veitur,1987,0000,None,0.0,91,257.0,None,None,BRS,ABA,762998400000,1589445623000,Orkuveita Reykjavíkur,10.0,None,4,NaN,403.0,1,None,None,None,2,1,5-6,Leiðhamrar 23,112,18.741724,33.0,None,1200.0,475.0,210.0,34.0,16,Skráður,EV,-2208988800000,1588710100000,1588710100000,"Nei, LUKOR",362783.073,407323.430,{C8483856-DC1F-4862-AB22-270A0CE429FC},Innfært,None,None,None,None,None,4360,None,None,None,None,None,None,NaN,ANNA,1.589446e+12,None,None,None,None,None,None,None,None,None,None,None,None,http://dmm.veitur.is/handlers/legacy/eign_opin...,E.25-12342,362783.073,407323.430
2,324290599,Tengiskápur,KSIP 463,659,659.0,12533,Notendakort,None,Veitur,1987,0000,None,0.0,91,225.0,None,None,BRS,ABA,762998400000,1589455082000,Orkuveita Reykjavíkur,10.0,None,4,NaN,403.0,1,None,None,None,1,1,4,Salthamrar 14,112,16.375075,63.0,None,1200.0,850.0,210.0,58.0,18,Skráður,EV,-2208988800000,1589314901000,1589314901000,"Nei, LUKOR",363165.131,407251.352,{0890A4C3-72BA-4D23-825B-56BE65BDFF09},Innfært,None,None,None,None,None,4474,None,None,None,None,None,None,NaN,ANNA,1.589455e+12,None,None,None,None,None,None,None,None,None,None,None,None,http://dmm.veitur.is/handlers/legacy/eign_opin...,E.25-12342,363165.131,407251.352
3,324290602,Tengiskápur,Óþekkt,659,659.0,12527,Notendakort,None,Veitur,1987,0000,None,0.0,91,225.0,None,None,BRS,BRS,762998400000,1612522243000,Orkuveita Reykjavíkur,10.0,None,4,NaN,403.0,1,None,None,None,1,1,8,Sporhamrar 8,112,29.153889,NaN,None,NaN,NaN,NaN,NaN,?,Skráður,EV,-2208988800000,1594393233000,1594393233000,Já,363074.797,407111.205,{A3F4F1FF-D14E-425F-9C84-3A731C079C75},Innfært,None,None,None,None,None,4468,None,None,None,None,None,None,NaN,BRS,1.612522e+12,None,None,None,None,None,None,None,None,None,None,None,None,http://dmm.veitur.is/handlers/legacy/eign_opin...,E.25-12342,363074.797,407111.205
4,324290603,Tengiskápur,Óþekkt,659,659.0,12550,Notendakort,None,Veitur,1987,0000,98-STAÐ,13845.0,91,135.0,None,None,BRS,BRS,897436800000,1612522248000,Orkuveita Reykjavíkur,10.0,None,4,NaN,403.0,1,None,None,None,1,1,8,Sporhamrar 8,112,16.908907,NaN,None,NaN,NaN,NaN,NaN,?,Skráður,EV,-2208988800000,1582023509000,1582023509000,Já,363051.983,407129.543,{E7560479-15FD-424C-9692-34041D08248D},Innfært,None,None,None,None,None,5662,None,None,Non